In [2]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import robot_vlp.data_collection.experment_processing as p
import robot_vlp.data_collection.communication as c
import pickle

from robot_vlp.config import EXPERIMENT_DATA_DIR


%load_ext autoreload
%autoreload 2

2024-12-16 17:12:36.610 | INFO     | robot_vlp.config:<module>:11 - PROJ_ROOT path is: /Users/tyrelglass/PhD/Repositories/robot-vlp


In [ ]:
import csv
import time



def read_vive_matrix(vive, n_readings = 10 ):
    readings = []
    for _ in range(n_readings):
        readings.append(np.array(vive.devices["tracker_1"].get_pose_matrix()))
        time.sleep(0.1)
    # mean_readings = np.mean(readings, axis = 0)
    return readings

def read_vive_radians(vive, n_readings = 10 ):
    readings = []
    for _ in range(n_readings):
        readings.append(np.array(vive.devices["tracker_1"].get_pose_radians()))
        time.sleep(0.1)
    # mean_readings = np.mean(readings, axis = 0)
    return readings

def read_vive_euler(vive, n_readings = 10 ):
    readings = []
    for _ in range(n_readings):
        readings.append(np.array(vive.devices["tracker_1"].get_pose_euler()))
        time.sleep(0.1)
    # mean_readings = np.mean(readings, axis = 0)
    return readings
    


def vive_robot_log_clear(log_file = 'robot_vive_data_log.csv' ):
    """Clears the log file and writes headers with a pipe delimiter."""
    with open(log_file, 'w') as f:
        f.write("vive_matrix|vive_radians|vive_euler|last_cmd\n")


def vive_robot_log_write(vive_matrix,vive_radians, vive_euler, cmd, log_file ):
    """Writes a single data point to the log file."""
    vive_matrix = str(vive_matrix).replace('\n', '')  # Remove newline characters
    vive_radians = str(vive_radians).replace('\n', '')  # Remove newline characters
    vive_euler = str(vive_euler).replace('\n', '')  # Remove newline characters


    last_cmd = str(cmd).replace('\n', '')    # Remove newline characters
    
    with open(log_file, 'a', newline='') as f:
        writer = csv.writer(f, delimiter='|', quotechar='"', quoting=csv.QUOTE_MINIMAL)
        writer.writerow([vive_matrix,vive_radians, vive_euler, cmd])


def take_vive_cal_point(point_no, log_file, vive, raw = True):
    vive_matrix = read_vive_matrix(vive)
    vive_radians = read_vive_radians(vive)
    vive_euler = read_vive_euler(vive)
    cmd = 'CAL:'+str(point_no)
    vive_robot_log_write(vive_matrix,vive_radians, vive_euler, cmd, log_file)

In [ ]:
vive_robot_log_clear('vive_test.csv')

In [ ]:
vive = c.vive_setup()

In [ ]:
c.read_vive(vive)

In [ ]:
take_vive_cal_point(12, 'vive_test.csv', vive)

## Analysis

In [3]:
df= pd.read_csv('vive_test.csv', delimiter = '|')


In [4]:
df

,vive_matrix,vive_radians,vive_euler,last_cmd
0,"[array(([[-0.45104212, 0.8916793 , -0.0383259...","[array([-0.20408764, -0.8230381 , -0.27322119,...","[array([-2.04085454e-01, -8.22990417e-01, -2.7...",CAL:1
1,"[array(([[-0.45058975, 0.8918261 , -0.0401871...","[array([ 0.54660803, -0.80820566, -0.76014632,...","[array([ 0.54661208, -0.80808544, -0.760002...",CAL:2
2,"[array(([[-0.45538935, 0.88908726, -0.0463062...","[array([ 0.06133102, -0.80049157, -1.52072322,...","[array([ 6.13817014e-02, -8.00429940e-01, -1.5...",CAL:3
3,"[array(([[-0.43959472, 0.8971468 , -0.0434062...","[array([-0.07183339, -0.82022434, -0.88942647,...","[array([-7.18419552e-02, -8.20292473e-01, -8.8...",CAL:4
4,"[array(([[ 0.28063598, 0.95894367, -0.0408713...","[array([-0.07262463, -0.82275653, -0.89183575,...","[array([-7.25184157e-02, -8.23029935e-01, -8.9...",CAL:5
5,"[array(([[ 0.8161006 , 0.5767266 , -0.0369631...","[array([-0.07280481, -0.82527298, -0.89501613,...","[array([-7.30121732e-02, -8.25124681e-01, -8.9...",CAL:6
6,"[array(([[ 0.94906586, -0.313453 , -0.0319566...","[array([-0.07225459, -0.82274789, -0.8924523 ,...","[array([-7.25096241e-02, -8.22699547e-01, -8.9...",CAL:7
7,"[array(([[ 0.6098656 , -0.7918774 , -0.0315304...","[array([-0.07206035, -0.82030696, -0.88915002,...","[array([-7.24374205e-02, -8.20199311e-01, -8.8...",CAL:8
8,"[array(([[-0.04232886, -0.9987538 , -0.0264405...","[array([-0.07325187, -0.82046187, -0.88941586,...","[array([-7.31331035e-02, -8.20721686e-01, -8.8...",CAL:9
9,"[array(([[-0.8361809 , -0.54755735, -0.0313431...","[array([-0.07273464, -0.81951874, -0.88809115,...","[array([-7.24799484e-02, -8.19393516e-01, -8.8...",CAL:10


In [6]:
df['vive_matrix'][0]

"[array(([[-0.45104212,  0.8916793 , -0.03832596, -0.20417914], [ 0.0133661 , -0.03618869, -0.99925554, -0.8230368 ], [-0.89240247, -0.45121866,  0.00440442, -0.27328077]],),      dtype=[('m', '<f4', (3, 4))]), array(([[-0.45096698,  0.8917247 , -0.03815338, -0.20407332], [ 0.01325621, -0.03605043, -0.999262  , -0.82293713], [-0.8924421 , -0.45114   ,  0.00443673, -0.2732079 ]],),      dtype=[('m', '<f4', (3, 4))]), array(([[-0.45084757,  0.8918003 , -0.0377961 , -0.2040732 ], [ 0.01320742, -0.03567409, -0.99927616, -0.823041  ], [-0.89250314, -0.45102048,  0.00430527, -0.27315807]],),      dtype=[('m', '<f4', (3, 4))]), array(([[-0.45099416,  0.8917351 , -0.03758545, -0.20411946], [ 0.0131308 , -0.03547756, -0.99928415, -0.82315683], [-0.89243025, -0.45116493,  0.00429105, -0.2733154 ]],),      dtype=[('m', '<f4', (3, 4))]), array(([[-0.45118573,  0.89163786, -0.03759271, -0.20412025], [ 0.01319249, -0.03545553, -0.99928415, -0.8230796 ], [-0.8923325 , -0.45135877,  0.00423417, -0.273

In [ ]:
def parse_vive(df):
        def check_for_none_array(v):
                if len(v[0].reshape(-1)) == 1:
                        return np.nan
                else:
                        return v
        df['vive_data'] = df['vive_data'].apply( lambda v: np.array(eval(v.replace('array','np.array'))))
        df['vive_data'] = df['vive_data'].apply(check_for_none_array) #also pull out only x, y, z

        row_filt = df['vive_data'].notna()
        df.loc[row_filt, 'vive_data'] = df['vive_data'][row_filt].apply(lambda v: np.apply_along_axis(average_vive_readings, 0,v))

        return df

In [ ]:
# df = c.process_cnc(df)

df = c.parse_vive(df.iloc[3:])
new_pose = c.transform_vive_df(df)

plt.plot(df['vive_x'], df['vive_y'])

In [ ]:
df= pd.read_csv('vive_test.csv', delimiter = '|')
df = c.parse_vive(df)
df = c.transform_vive_df(df)

plt.plot(df['vive_yaw'], label = 'yaw', marker = '.')
plt.plot(df['vive_pitch'], label = 'pitch', marker = '.')
plt.plot(df['vive_roll'], label = 'roll', marker = '.')
plt.legend()

In [ ]:
plt.plot(df['vive_yaw']%360, label = 'yaw', marker = '.')
plt.plot(df['vive_pitch'], label = 'pitch', marker = '.')
plt.plot(df['vive_roll'], label = 'roll', marker = '.')
# plt.plot(df['vive_heading'], label = 'roll', marker = '.')
plt.legend()

In [ ]:
vive = np.array(df['vive_data'].to_list())

# plt.plot(vive[:,3])
plt.plot(vive[:7,4], marker = '.')
# plt.plot(vive[:,5])